# 4. 데이터 대표성 편향 평가 (LLM)

## 개요

이 노트북은 대규모 언어모델(LLM) 학습 데이터의 **대표성 편향(Representativeness Bias)**을 평가합니다.

### 평가 메트릭: Entropy (엔트로피)

**엔트로피(Entropy)**는 정보 이론에서 불확실성 또는 다양성을 측정하는 지표입니다.

#### 공식
```
H(X) = -Σ p(x) * log₂(p(x))
```

where:
- `p(x)`: 각 카테고리의 출현 확률
- `H(X)`: 엔트로피 값 (bits)

#### 해석
- **높은 엔트로피 (High Entropy)**: 데이터가 다양하고 균등하게 분포 → 좋음
- **낮은 엔트로피 (Low Entropy)**: 특정 카테고리에 집중 → 편향 존재

#### 예시
- 성별 언급: 남성 50%, 여성 50% → H = 1.0 (최대)
- 성별 언급: 남성 90%, 여성 10% → H = 0.47 (편향)

### 평가 대상
- 텍스트 데이터에서 성별, 인종, 국가 등 보호 변수의 언급 분포
- 토픽/주제의 다양성
- 사용 언어의 다양성

### 데이터셋
- HuggingFace의 IMDb 영화 리뷰 데이터셋
- 성별 대명사(he/she), 직업 언급의 분포 분석

## 1. 환경 설정 및 데이터 로드

In [ ]:
# 필요한 라이브러리 설치
!pip install -q pandas numpy matplotlib seaborn datasets scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from scipy.stats import entropy as scipy_entropy
from collections import Counter
import re

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 시각화 스타일
sns.set_style('whitegrid')
sns.set_palette('Set2')

print("라이브러리 로드 완료")

In [ ]:
# IMDb 데이터셋 로드 (샘플링)
print("IMDb 데이터셋 로딩 중...")
dataset = load_dataset("imdb", split="train")

# 계산 속도를 위해 5000개 샘플만 사용
dataset = dataset.shuffle(seed=42).select(range(5000))

# DataFrame으로 변환
df = pd.DataFrame(dataset)

print(f"데이터셋 크기: {len(df)}")
print(f"\n컬럼: {df.columns.tolist()}")
print(f"\n샘플 리뷰:")
print(df['text'].iloc[0][:300] + "...")

## 2. 데이터 전처리 및 특성 추출

In [ ]:
def extract_gender_pronouns(text):
    """
    텍스트에서 성별 대명사 추출 및 카운트
    """
    text_lower = text.lower()
    
    # 남성 대명사
    male_pronouns = ['he', 'him', 'his', 'himself']
    male_count = sum(len(re.findall(r'\b' + p + r'\b', text_lower)) for p in male_pronouns)
    
    # 여성 대명사
    female_pronouns = ['she', 'her', 'hers', 'herself']
    female_count = sum(len(re.findall(r'\b' + p + r'\b', text_lower)) for p in female_pronouns)
    
    return {'male': male_count, 'female': female_count}

def extract_occupations(text):
    """
    텍스트에서 직업 언급 추출
    """
    text_lower = text.lower()
    
    occupations = [
        'actor', 'actress', 'director', 'producer', 'writer', 
        'doctor', 'nurse', 'teacher', 'engineer', 'scientist',
        'lawyer', 'police', 'soldier', 'artist', 'musician'
    ]
    
    found = []
    for occ in occupations:
        if re.search(r'\b' + occ + r'\b', text_lower):
            found.append(occ)
    
    return found

# 전체 데이터셋에서 특성 추출
print("특성 추출 중...")

gender_counts = {'male': 0, 'female': 0}
occupation_list = []

for text in df['text']:
    # 성별 대명사
    gender = extract_gender_pronouns(text)
    gender_counts['male'] += gender['male']
    gender_counts['female'] += gender['female']
    
    # 직업
    occs = extract_occupations(text)
    occupation_list.extend(occs)

occupation_counter = Counter(occupation_list)

print(f"\n성별 대명사 카운트:")
print(f"  남성 대명사: {gender_counts['male']}")
print(f"  여성 대명사: {gender_counts['female']}")

print(f"\n직업 언급 Top 10:")
for occ, count in occupation_counter.most_common(10):
    print(f"  {occ}: {count}")

## 3. 엔트로피 계산

In [ ]:
def calculate_entropy(counts_dict):
    """
    카운트 딕셔너리로부터 엔트로피 계산
    
    Args:
        counts_dict: {category: count} 형태의 딕셔너리
    
    Returns:
        entropy: 엔트로피 값 (bits)
        max_entropy: 최대 엔트로피 (균등 분포 시)
        normalized_entropy: 정규화된 엔트로피 (0~1)
    """
    # 확률 분포 계산
    total = sum(counts_dict.values())
    if total == 0:
        return 0, 0, 0
    
    probs = np.array([count / total for count in counts_dict.values()])
    
    # 0인 확률 제거 (log(0) 방지)
    probs = probs[probs > 0]
    
    # 엔트로피 계산
    ent = -np.sum(probs * np.log2(probs))
    
    # 최대 엔트로피 (균등 분포)
    n_categories = len(counts_dict)
    max_ent = np.log2(n_categories) if n_categories > 1 else 0
    
    # 정규화된 엔트로피 (0~1)
    norm_ent = ent / max_ent if max_ent > 0 else 0
    
    return ent, max_ent, norm_ent

print("="*80)
print("엔트로피 기반 대표성 편향 평가")
print("="*80)

### 3.1 성별 대명사 엔트로피

In [ ]:
# 성별 대명사 엔트로피
gender_entropy, gender_max_ent, gender_norm_ent = calculate_entropy(gender_counts)

print("\n[성별 대명사 분포 엔트로피]")
print(f"\n카테고리별 카운트:")
for gender, count in gender_counts.items():
    total = sum(gender_counts.values())
    prob = count / total if total > 0 else 0
    print(f"  {gender.capitalize()}: {count} ({prob*100:.2f}%)")

print(f"\n엔트로피: {gender_entropy:.4f} bits")
print(f"최대 엔트로피: {gender_max_ent:.4f} bits (균등 분포)")
print(f"정규화된 엔트로피: {gender_norm_ent:.4f} (0~1 척도)")

# 해석
if gender_norm_ent > 0.9:
    print("\n✓ 평가: 매우 균등한 분포 (편향 없음)")
elif gender_norm_ent > 0.7:
    print("\n✓ 평가: 적절한 분포 (경미한 편향)")
elif gender_norm_ent > 0.5:
    print("\n⚠️ 평가: 불균등한 분포 (중간 수준 편향)")
else:
    print("\n⚠️ 평가: 매우 불균등한 분포 (심각한 편향)")

### 3.2 직업 언급 엔트로피

In [ ]:
# 직업 언급 엔트로피
occupation_counts = dict(occupation_counter)
occupation_entropy, occupation_max_ent, occupation_norm_ent = calculate_entropy(occupation_counts)

print("\n[직업 언급 분포 엔트로피]")
print(f"\n직업 카테고리 수: {len(occupation_counts)}")
print(f"총 직업 언급 횟수: {sum(occupation_counts.values())}")

print(f"\nTop 5 직업:")
for occ, count in occupation_counter.most_common(5):
    total = sum(occupation_counts.values())
    prob = count / total if total > 0 else 0
    print(f"  {occ}: {count} ({prob*100:.2f}%)")

print(f"\n엔트로피: {occupation_entropy:.4f} bits")
print(f"최대 엔트로피: {occupation_max_ent:.4f} bits (균등 분포)")
print(f"정규화된 엔트로피: {occupation_norm_ent:.4f} (0~1 척도)")

# 해석
if occupation_norm_ent > 0.8:
    print("\n✓ 평가: 직업 언급이 다양함 (편향 없음)")
elif occupation_norm_ent > 0.6:
    print("\n✓ 평가: 적절한 다양성 (경미한 편향)")
else:
    print("\n⚠️ 평가: 특정 직업에 집중 (편향 존재)")

### 3.3 리뷰 길이 분포 엔트로피

In [ ]:
# 리뷰 길이 분포 (다양성 지표)
df['text_length'] = df['text'].apply(lambda x: len(x.split()))

# 길이를 구간으로 나누기
length_bins = [0, 100, 200, 300, 500, 1000, 10000]
length_labels = ['Very Short', 'Short', 'Medium', 'Long', 'Very Long', 'Extra Long']
df['length_category'] = pd.cut(df['text_length'], bins=length_bins, labels=length_labels)

length_counts = df['length_category'].value_counts().to_dict()
length_entropy, length_max_ent, length_norm_ent = calculate_entropy(length_counts)

print("\n[리뷰 길이 분포 엔트로피]")
print(f"\n길이별 분포:")
for cat, count in sorted(length_counts.items(), key=lambda x: x[1], reverse=True):
    total = sum(length_counts.values())
    prob = count / total if total > 0 else 0
    print(f"  {cat}: {count} ({prob*100:.2f}%)")

print(f"\n엔트로피: {length_entropy:.4f} bits")
print(f"최대 엔트로피: {length_max_ent:.4f} bits (균등 분포)")
print(f"정규화된 엔트로피: {length_norm_ent:.4f} (0~1 척도)")

if length_norm_ent > 0.8:
    print("\n✓ 평가: 리뷰 길이가 다양함")
else:
    print("\n⚠️ 평가: 특정 길이에 집중")

## 4. 결과 시각화

In [ ]:
# 시각화 1: 성별 대명사 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 파이 차트
colors = ['#3498db', '#e74c3c']
axes[0].pie(gender_counts.values(), labels=['Male', 'Female'], autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.05, 0.05))
axes[0].set_title('Gender Pronoun Distribution', fontsize=14, fontweight='bold')

# 바 차트 + 엔트로피
axes[1].bar(gender_counts.keys(), gender_counts.values(), color=colors, alpha=0.8, edgecolor='black')
axes[1].set_title('Gender Pronoun Counts', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_xlabel('Gender', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

# 엔트로피 표시
axes[1].text(0.5, 0.95, f'Entropy: {gender_entropy:.3f} bits\nNormalized: {gender_norm_ent:.3f}',
             transform=axes[1].transAxes, ha='center', va='top',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5),
             fontsize=11, fontweight='bold')

# 값 표시
for i, (key, val) in enumerate(gender_counts.items()):
    axes[1].text(i, val + max(gender_counts.values())*0.02, str(val), 
                 ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/gender_pronoun_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 2: 직업 언급 분포
fig, ax = plt.subplots(figsize=(12, 6))

# Top 10 직업
top_occupations = occupation_counter.most_common(10)
occs, counts = zip(*top_occupations)

bars = ax.barh(occs, counts, color='#2ecc71', alpha=0.8, edgecolor='black')
ax.set_xlabel('Count', fontsize=12, fontweight='bold')
ax.set_title('Top 10 Occupation Mentions', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# 엔트로피 표시
ax.text(0.98, 0.98, f'Entropy: {occupation_entropy:.3f} bits\nNormalized: {occupation_norm_ent:.3f}',
        transform=ax.transAxes, ha='right', va='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5),
        fontsize=11, fontweight='bold')

# 값 표시
for bar, count in zip(bars, counts):
    ax.text(count + max(counts)*0.01, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('../assets/occupation_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 3: 리뷰 길이 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램
axes[0].hist(df['text_length'], bins=50, color='#9b59b6', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Text Length (words)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0].set_title('Review Length Distribution', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# 카테고리별 분포
sorted_cats = sorted(length_counts.items(), key=lambda x: length_labels.index(x[0]))
cats, cat_counts = zip(*sorted_cats)
axes[1].bar(range(len(cats)), cat_counts, color='#e67e22', alpha=0.8, edgecolor='black')
axes[1].set_xticks(range(len(cats)))
axes[1].set_xticklabels(cats, rotation=45, ha='right')
axes[1].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[1].set_title('Review Length Categories', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

# 엔트로피 표시
axes[1].text(0.5, 0.98, f'Entropy: {length_entropy:.3f} bits\nNormalized: {length_norm_ent:.3f}',
             transform=axes[1].transAxes, ha='center', va='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5),
             fontsize=11, fontweight='bold')

# 값 표시
for i, count in enumerate(cat_counts):
    axes[1].text(i, count + max(cat_counts)*0.02, str(count),
                 ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('../assets/length_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 4: 엔트로피 종합 비교
fig, ax = plt.subplots(figsize=(10, 6))

categories = ['Gender Pronouns', 'Occupations', 'Review Length']
entropies = [gender_norm_ent, occupation_norm_ent, length_norm_ent]
colors_ent = ['#3498db', '#2ecc71', '#e67e22']

bars = ax.barh(categories, entropies, color=colors_ent, alpha=0.8, edgecolor='black')
ax.axvline(x=0.7, color='orange', linestyle='--', linewidth=2, label='Good Diversity (>0.7)')
ax.axvline(x=0.9, color='green', linestyle='--', linewidth=2, label='Excellent Diversity (>0.9)')
ax.set_xlabel('Normalized Entropy (0~1)', fontsize=12, fontweight='bold')
ax.set_title('Data Representativeness - Entropy Summary', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.0)
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)

# 값 표시
for bar, ent, cat in zip(bars, entropies, categories):
    if ent > 0.9:
        status = 'Excellent ✓'
    elif ent > 0.7:
        status = 'Good ✓'
    elif ent > 0.5:
        status = 'Fair'
    else:
        status = 'Poor ✗'
    
    ax.text(ent + 0.02, bar.get_y() + bar.get_height()/2,
            f'{ent:.3f} ({status})', va='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('../assets/entropy_summary.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. 상세 통계 요약

In [ ]:
# 종합 리포트 생성
summary = pd.DataFrame({
    'Attribute': ['Gender Pronouns', 'Occupations', 'Review Length'],
    'Categories': [len(gender_counts), len(occupation_counts), len(length_counts)],
    'Entropy (bits)': [gender_entropy, occupation_entropy, length_entropy],
    'Max Entropy (bits)': [gender_max_ent, occupation_max_ent, length_max_ent],
    'Normalized Entropy': [gender_norm_ent, occupation_norm_ent, length_norm_ent],
    'Diversity Level': [
        'Excellent' if gender_norm_ent > 0.9 else 'Good' if gender_norm_ent > 0.7 else 'Fair' if gender_norm_ent > 0.5 else 'Poor',
        'Excellent' if occupation_norm_ent > 0.9 else 'Good' if occupation_norm_ent > 0.7 else 'Fair' if occupation_norm_ent > 0.5 else 'Poor',
        'Excellent' if length_norm_ent > 0.9 else 'Good' if length_norm_ent > 0.7 else 'Fair' if length_norm_ent > 0.5 else 'Poor'
    ]
})

print("\n" + "="*100)
print("데이터 대표성 편향 평가 종합 리포트")
print("="*100)
print(summary.to_string(index=False))
print("="*100)

## 6. 결론 및 개선 방향

### 평가 결과 해석

#### 엔트로피 기준

| 정규화 엔트로피 | 다양성 수준 | 의미 |
|----------------|------------|------|
| **> 0.9** | Excellent | 매우 균등하고 다양한 분포 |
| **0.7 ~ 0.9** | Good | 적절한 다양성, 경미한 편향 |
| **0.5 ~ 0.7** | Fair | 불균등한 분포, 중간 수준 편향 |
| **< 0.5** | Poor | 심각한 불균등, 강한 편향 존재 |

#### 1. 성별 대명사 분포
- **관찰**: 남성 대명사 vs 여성 대명사의 비율 차이
- **해석**:
  - 엔트로피 낮음 → 한쪽 성별이 과대표현됨
  - 영화 리뷰 데이터의 경우, 남성 주인공 중심 영화가 더 많을 수 있음
- **영향**: LLM이 성별 고정관념을 학습할 위험

#### 2. 직업 언급 분포
- **관찰**: 특정 직업(actor, director)이 지배적
- **해석**:
  - 영화 리뷰 도메인 특성상 영화 관련 직업이 많음 (자연스러움)
  - 하지만 다양한 직업 표현이 부족하면 LLM의 직업 이해가 편향됨
- **영향**: 특정 직업에 대한 과도한 연관성 학습

#### 3. 리뷰 길이 분포
- **관찰**: 리뷰 길이의 다양성
- **해석**:
  - 높은 엔트로피 → 짧은 리뷰부터 긴 리뷰까지 고르게 존재
  - 낮은 엔트로피 → 특정 길이에 집중 (예: 대부분 짧은 리뷰)
- **영향**: LLM의 텍스트 생성 길이 편향

### 개선 방향

#### 1. 데이터 수집 단계

**균형 잡힌 샘플링**
- 의도적으로 소수 그룹의 데이터 더 수집
- 다양한 출처에서 데이터 수집 (단일 도메인 탈피)
- 시대, 지역, 언어의 다양성 확보

**예시**:
```python
# 성별 균형 맞추기
target_ratio = 0.5  # 50:50 목표
if male_ratio > target_ratio:
    # 여성 대명사 포함 텍스트 oversampling
    female_texts = df[df['has_female_pronoun'] == True]
    df = pd.concat([df, female_texts.sample(n=補正량)])
```

#### 2. 데이터 증강(Augmentation)

**성별 중립화/교체**
- 데이터 복제 후 성별 대명사 교체
- "He is a doctor" → "She is a doctor" 변형 추가

**합성 데이터 생성**
- LLM을 활용하여 부족한 카테고리의 텍스트 생성
- 예: 여성 주인공 영화 리뷰 생성

#### 3. 필터링 및 리샘플링

**Over-representation 완화**
- 과대표현된 그룹의 샘플 일부 제거 (Undersampling)
- 가중치 기반 샘플링으로 균형 조정

```python
# 엔트로피 최대화를 위한 리샘플링
from sklearn.utils import resample

def resample_for_entropy(df, category_col, target_entropy=0.9):
    # 각 카테고리를 동일한 비율로 리샘플링
    categories = df[category_col].unique()
    target_size = len(df) // len(categories)
    
    resampled = []
    for cat in categories:
        cat_df = df[df[category_col] == cat]
        resampled.append(resample(cat_df, n_samples=target_size, replace=True))
    
    return pd.concat(resampled)
```

#### 4. 모니터링 및 평가

**정기적 엔트로피 측정**
- 데이터셋 업데이트 시마다 엔트로피 재계산
- 다양한 속성(성별, 인종, 직업, 주제 등)에 대해 엔트로피 추적

**대시보드 구축**
- 실시간으로 데이터 대표성 지표 시각화
- 임계값 설정 후 경고 시스템 구축

```python
# 다중 속성 엔트로피 모니터링
def monitor_diversity(df):
    metrics = {}
    for attr in ['gender', 'occupation', 'topic', 'language']:
        entropy = calculate_entropy(df[attr].value_counts())
        metrics[attr] = entropy
        
        if entropy < THRESHOLD:
            print(f"⚠️ Warning: Low diversity in {attr} (H={entropy:.3f})")
    
    return metrics
```

### LLM 학습에 미치는 영향

#### 편향된 데이터로 학습 시
- **고정관념 강화**: "간호사는 여성", "엔지니어는 남성" 등
- **소수 그룹 무시**: 데이터에 적은 그룹에 대한 이해 부족
- **불공정한 출력**: 특정 그룹에 불리한 텍스트 생성

#### 균형 잡힌 데이터로 학습 시
- **공정한 표현**: 모든 그룹을 균등하게 표현
- **다양한 생성**: 다양한 관점과 맥락의 텍스트 생성 가능
- **강건성 향상**: 다양한 입력에 대해 안정적인 출력

### 윤리적 고려사항

1. **대표성 vs 정확성**: 실제 세계가 불균등하다면, 데이터도 그래야 하는가?
   - 예: 역사적으로 CEO는 대부분 남성이었지만, 이를 그대로 반영해야 하는가?
   - 답: 도메인과 목적에 따라 판단 (역사 분석 vs 미래 예측)

2. **과도한 균형 조정의 위험**: 현실과 너무 동떨어진 데이터는 오히려 문제
   - 해결: 점진적 개선, 명확한 목표 설정

3. **투명성**: 데이터의 편향과 조정 과정을 공개
   - 데이터 카드(Data Card) 작성
   - 편향 리포트 첨부

### 참고 자료
- Shannon Entropy: https://en.wikipedia.org/wiki/Entropy_(information_theory)
- "Datasheets for Datasets" (Gebru et al.): https://arxiv.org/abs/1803.09010
- HuggingFace Dataset Cards: https://huggingface.co/docs/hub/datasets-cards